In [0]:
# --- Can you have the genie re organize columns ( logical order)?  updated

# -- Can you add a historical to the dash so we can see balance and rate trends by product and subproduct?  updated
# Can you add drill downs to dashbaord that that issances and its type etc...

In [0]:
# --Account numbers 
# --Add source  (FIS, JH, Man)
# --Add Sec IDs
# --Add counterparty type 


# --Only render days where the last business day of the month = the asofdate

# --

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as f
import re

# Output configuration widgets
dbutils.widgets.dropdown("view_type", "temp view", ["table", "temp view"], "1. View Type")
dbutils.widgets.dropdown("write_mode", "overwrite", ["overwrite", "append"], "2. Write Mode")
dbutils.widgets.dropdown("environment", "dev", ["prod", "dev"], "3. Environment")
dbutils.widgets.text("view_name", "wholesalefunding", "4. View Name")
dbutils.widgets.text(
    "source_folder",
    "/Workspace/Users/joshuawaller@synovus.com/liquidity/wholesale_funding/FlatFlies",
    "5. Source Folder (Workspace Path)",
)
dbutils.widgets.dropdown("save_html", "yes", ["yes", "no"], "6. Save HTML Report")
dbutils.widgets.dropdown("save_csv", "yes", ["yes", "no"], "7. Save CSV Report")

view_type = dbutils.widgets.get("view_type")
write_mode = dbutils.widgets.get("write_mode")
environment = dbutils.widgets.get("environment")
view_name = dbutils.widgets.get("view_name").strip().lower()
save_html = dbutils.widgets.get("save_html") == "yes"
save_csv = dbutils.widgets.get("save_csv") == "yes"

# Report output folders
html_output_folder = "/Workspace/Users/joshuawaller@synovus.com/liquidity/wholesale_funding/Outputs/html"
csv_output_folder = "/Workspace/Users/joshuawaller@synovus.com/liquidity/wholesale_funding/Outputs/csv"

# Normalize the source folder: drop any 'file:' prefix and trailing slash, ensure '/Workspace' root
source_folder = dbutils.widgets.get("source_folder").strip()
source_folder = re.sub(r"^file:", "", source_folder).rstrip("/")
if not source_folder.startswith("/Workspace"):
    source_folder = "/Workspace" + (source_folder if source_folder.startswith("/") else "/" + source_folder)

# Validate the view name (letters, numbers, underscores only)
if not re.fullmatch(r"[a-z_][a-z0-9_]*", view_name):
    raise ValueError(f"Invalid view name: '{view_name}'. Use letters, numbers and underscores only.")

# Resolve destination: prod -> finance.reporting, dev -> finance.dev
target_schema = {"prod": "finance.reporting", "dev": "finance.dev"}[environment]

# Tables are written to Unity Catalog; temp views are session-scoped and use the bare name
target_name = f"{target_schema}.{view_name}" if view_type == "table" else view_name

print(f"View type:   {view_type}")
print(f"Write mode:  {write_mode}")
print(f"Environment: {environment}")
print(f"Target:      {target_name}")
print(f"Source:      {source_folder}")
print(f"Save HTML:   {'yes -> ' + html_output_folder if save_html else 'no'}")
print(f"Save CSV:    {'yes -> ' + csv_output_folder if save_csv else 'no'}")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_holidays AS
SELECT * FROM VALUES
  (2026, 'New Year''s Day',                  DATE '2026-01-01'),
  (2026, 'Martin Luther King Jr. Day',       DATE '2026-01-19'),
  (2026, 'Washington''s Birthday',           DATE '2026-02-16'),
  (2026, 'Memorial Day',                     DATE '2026-05-25'),
  (2026, 'Juneteenth',                       DATE '2026-06-19'),
  (2026, 'Independence Day (observed)',      DATE '2026-07-03'),
  (2026, 'Labor Day',                        DATE '2026-09-07'),
  (2026, 'Columbus Day',                     DATE '2026-10-12'),
  (2026, 'Veterans Day',                     DATE '2026-11-11'),
  (2026, 'Thanksgiving',                     DATE '2026-11-26'),
  (2026, 'Christmas Day',                    DATE '2026-12-25'),
  (2027, 'New Year''s Day',                  DATE '2027-01-01'),
  (2027, 'Martin Luther King Jr. Day',       DATE '2027-01-18'),
  (2027, 'Washington''s Birthday',           DATE '2027-02-15'),
  (2027, 'Memorial Day',                     DATE '2027-05-31'),
  (2027, 'Juneteenth (observed)',            DATE '2027-06-18'),
  (2027, 'Independence Day (observed)',      DATE '2027-07-05'),
  (2027, 'Labor Day',                        DATE '2027-09-06'),
  (2027, 'Columbus Day',                     DATE '2027-10-11'),
  (2027, 'Veterans Day',                     DATE '2027-11-11'),
  (2027, 'Thanksgiving',                     DATE '2027-11-25'),
  (2027, 'Christmas Day (observed)',         DATE '2027-12-24'),
  (2028, 'New Year''s Day',                  DATE '2028-01-01'),
  (2028, 'Martin Luther King Jr. Day',       DATE '2028-01-17'),
  (2028, 'Washington''s Birthday',           DATE '2028-02-21'),
  (2028, 'Memorial Day',                     DATE '2028-05-29'),
  (2028, 'Juneteenth',                       DATE '2028-06-19'),
  (2028, 'Independence Day',                DATE '2028-07-04'),
  (2028, 'Labor Day',                        DATE '2028-09-04'),
  (2028, 'Columbus Day',                     DATE '2028-10-09'),
  (2028, 'Veterans Day (observed)',          DATE '2028-11-10'),
  (2028, 'Thanksgiving',                     DATE '2028-11-23'),
  (2028, 'Christmas Day',                    DATE '2028-12-25'),
  (2029, 'New Year''s Day',                  DATE '2029-01-01'),
  (2029, 'Martin Luther King Jr. Day',       DATE '2029-01-15'),
  (2029, 'Washington''s Birthday',           DATE '2029-02-19'),
  (2029, 'Memorial Day',                     DATE '2029-05-28'),
  (2029, 'Juneteenth',                       DATE '2029-06-19'),
  (2029, 'Independence Day',                DATE '2029-07-04'),
  (2029, 'Labor Day',                        DATE '2029-09-03'),
  (2029, 'Columbus Day',                     DATE '2029-10-08'),
  (2029, 'Veterans Day (observed)',          DATE '2029-11-12'),
  (2029, 'Thanksgiving',                     DATE '2029-11-22'),
  (2029, 'Christmas Day',                    DATE '2029-12-25'),
  (2030, 'New Year''s Day',                  DATE '2030-01-01'),
  (2030, 'Martin Luther King Jr. Day',       DATE '2030-01-21'),
  (2030, 'Washington''s Birthday',           DATE '2030-02-18'),
  (2030, 'Memorial Day',                     DATE '2030-05-27'),
  (2030, 'Juneteenth',                       DATE '2030-06-19'),
  (2030, 'Independence Day',                DATE '2030-07-04'),
  (2030, 'Labor Day',                        DATE '2030-09-02'),
  (2030, 'Columbus Day',                     DATE '2030-10-14'),
  (2030, 'Veterans Day',                     DATE '2030-11-11'),
  (2030, 'Thanksgiving',                     DATE '2030-11-28'),
  (2030, 'Christmas Day',                    DATE '2030-12-25'),
  (2031, 'New Year''s Day',                  DATE '2031-01-01'),
  (2031, 'Martin Luther King Jr. Day',       DATE '2031-01-20'),
  (2031, 'Washington''s Birthday',           DATE '2031-02-17'),
  (2031, 'Memorial Day',                     DATE '2031-05-26'),
  (2031, 'Juneteenth',                       DATE '2031-06-19'),
  (2031, 'Independence Day',                DATE '2031-07-04'),
  (2031, 'Labor Day',                        DATE '2031-09-01'),
  (2031, 'Columbus Day',                     DATE '2031-10-13'),
  (2031, 'Veterans Day',                     DATE '2031-11-11'),
  (2031, 'Thanksgiving',                     DATE '2031-11-27'),
  (2031, 'Christmas Day',                    DATE '2031-12-25'),
  (2032, 'New Year''s Day',                  DATE '2032-01-01'),
  (2032, 'Martin Luther King Jr. Day',       DATE '2032-01-19'),
  (2032, 'Washington''s Birthday',           DATE '2032-02-16'),
  (2032, 'Memorial Day',                     DATE '2032-05-31'),
  (2032, 'Juneteenth',                       DATE '2032-06-19'),
  (2032, 'Independence Day',                DATE '2032-07-04'),
  (2032, 'Labor Day',                        DATE '2032-09-06'),
  (2032, 'Columbus Day',                     DATE '2032-10-11'),
  (2032, 'Veterans Day',                     DATE '2032-11-11'),
  (2032, 'Thanksgiving',                     DATE '2032-11-25'),
  (2032, 'Christmas Day',                    DATE '2032-12-25'),
  (2033, 'New Year''s Day',                  DATE '2033-01-01'),
  (2033, 'Martin Luther King Jr. Day',       DATE '2033-01-17'),
  (2033, 'Washington''s Birthday',           DATE '2033-02-21'),
  (2033, 'Memorial Day',                     DATE '2033-05-30'),
  (2033, 'Juneteenth',                       DATE '2033-06-19'),
  (2033, 'Independence Day',                DATE '2033-07-04'),
  (2033, 'Labor Day',                        DATE '2033-09-05'),
  (2033, 'Columbus Day',                     DATE '2033-10-10'),
  (2033, 'Veterans Day',                     DATE '2033-11-11'),
  (2033, 'Thanksgiving',                     DATE '2033-11-24'),
  (2033, 'Christmas Day',                    DATE '2033-12-25'),
  (2034, 'New Year''s Day',                  DATE '2034-01-01'),
  (2034, 'Martin Luther King Jr. Day',       DATE '2034-01-16'),
  (2034, 'Washington''s Birthday',           DATE '2034-02-20'),
  (2034, 'Memorial Day',                     DATE '2034-05-29'),
  (2034, 'Juneteenth',                       DATE '2034-06-19'),
  (2034, 'Independence Day',                DATE '2034-07-04'),
  (2034, 'Labor Day',                        DATE '2034-09-04'),
  (2034, 'Columbus Day',                     DATE '2034-10-09'),
  (2034, 'Veterans Day',                     DATE '2034-11-11'),
  (2034, 'Thanksgiving',                     DATE '2034-11-23'),
  (2034, 'Christmas Day',                    DATE '2034-12-25'),
  (2035, 'New Year''s Day',                  DATE '2035-01-01'),
  (2035, 'Martin Luther King Jr. Day',       DATE '2035-01-15'),
  (2035, 'Washington''s Birthday',           DATE '2035-02-19'),
  (2035, 'Memorial Day',                     DATE '2035-05-28'),
  (2035, 'Juneteenth',                       DATE '2035-06-19'),
  (2035, 'Independence Day',                DATE '2035-07-04'),
  (2035, 'Labor Day',                        DATE '2035-09-03'),
  (2035, 'Columbus Day',                     DATE '2035-10-08'),
  (2035, 'Veterans Day',                     DATE '2035-11-11'),
  (2035, 'Thanksgiving',                     DATE '2035-11-22'),
  (2035, 'Christmas Day',                    DATE '2035-12-25')
AS t(Year, Holiday, Holiday_Date);


CREATE OR REPLACE TEMP VIEW dim_dates  AS 

WITH DateRange AS (
    SELECT explode(
        sequence(
            to_date('2018-12-01'),
            to_date('2040-12-31'),
            interval 1 day
        )
    ) AS Date
)

SELECT
    d.Date,
    date_format(d.Date, 'MMM yyyy') AS Month_Year,
    date_format(d.Date, 'MMMM') AS Month_Name,
    day(d.Date) AS Day_of_Month,
    year(d.Date) AS Year,
    month(d.Date) AS Month_Number,
    dayofweek(d.Date) AS DayOfWeekNumber,
    date_format(d.Date, 'EEEE') AS Day_Name,
    CASE WHEN d.Date = current_date() THEN 1 ELSE 0 END AS IsToday,
    quarter(d.Date) AS Quarter,
    dayofyear(d.Date) AS Day_of_Year,
    weekofyear(d.Date) AS Week_of_Year,
    (day(d.Date) - 1) / 7 + 1 AS Week_of_Month,
    date_trunc('week', d.Date) AS Start_of_Week,
    date_add(date_trunc('week', d.Date), 6) AS End_of_Week,
    day(last_day(d.Date)) AS Days_in_Month,
    trunc(d.Date, 'MM') AS Start_of_Month,
    last_day(d.Date) AS End_of_Month,
    make_date(year(d.Date), (quarter(d.Date)-1)*3 + 1, 1) AS Start_of_Quarter,
    last_day(make_date(year(d.Date), (quarter(d.Date)-1)*3 + 3, 1)) AS End_of_Quarter,
    make_date(year(d.Date), 1, 1) AS Start_of_Year,
    make_date(year(d.Date), 12, 31) AS End_of_Year,
    datediff(make_date(year(d.Date), 12, 31), make_date(year(d.Date), 1, 1)) + 1 AS Days_in_Year,
    1 AS Day_Count,
    substr(date_format(d.Date, 'MMMM'), 1, 3) AS Month_Name_Short,
    concat('Q', quarter(d.Date), ' ', substr(year(d.Date), 3, 2)) AS Quarter_Year,
    CASE 
        WHEN month(d.Date) = month(current_date()) 
         AND year(d.Date) = year(current_date()) THEN 1 
        ELSE 0 
    END AS IsCurrentMonth,
    CASE WHEN h.Holiday_Date IS NOT NULL THEN 1 ELSE 0 END AS Is_Holiday,
    (
        SELECT max(bd.Date)
        FROM DateRange bd
        LEFT JOIN dim_holidays h2 ON bd.Date = h2.Holiday_Date
        WHERE month(bd.Date) = month(d.Date)
          AND year(bd.Date) = year(d.Date)
          AND dayofweek(bd.Date) NOT IN (1, 7) -- 1=Sunday, 7=Saturday
          AND h2.Holiday_Date IS NULL
          AND bd.Date <= last_day(d.Date)
    ) AS Last_Business_Day_Of_Month,
    (
        SELECT max(bd.Date)
        FROM DateRange bd
        LEFT JOIN dim_holidays h2 ON bd.Date = h2.Holiday_Date
        WHERE month(bd.Date) = month(add_months(d.Date, 1))
          AND year(bd.Date) = year(add_months(d.Date, 1))
          AND dayofweek(bd.Date) NOT IN (1, 7)
          AND h2.Holiday_Date IS NULL
          AND bd.Date <= last_day(add_months(d.Date, 1))
    ) AS Next_Last_Business_Day_Of_Month,
    (
        SELECT max(bd.Date)
        FROM DateRange bd
        LEFT JOIN dim_holidays h2 ON bd.Date = h2.Holiday_Date
        WHERE month(bd.Date) = month(add_months(d.Date, -1))
          AND year(bd.Date) = year(add_months(d.Date, -1))
          AND dayofweek(bd.Date) NOT IN (1, 7)
          AND h2.Holiday_Date IS NULL
          AND bd.Date <= last_day(add_months(d.Date, -1))
    ) AS Previous_Last_Business_Day_Of_Month,
    -- Most recent business day logic
    (
        SELECT max(bd.Date)
        FROM DateRange bd
        LEFT JOIN dim_holidays h2 ON bd.Date = h2.Holiday_Date
        WHERE bd.Date <= d.Date
          AND dayofweek(bd.Date) NOT IN (1, 7) -- 1=Sunday, 7=Saturday
          AND h2.Holiday_Date IS NULL
    ) AS most_recent_business_day
FROM DateRange d
LEFT JOIN dim_holidays h
  ON d.Date = h.Holiday_Date

In [0]:
id_lookup_data = [
    ("SNV-ARC-1", "SNV_bor_464000000000_1", "165.9998001"),
    ("SNV-ARC-2", "SNV_bor_464000000000_2", "165.9998001"),
    ("SNV-ARC-3", "SNV_bor_464000000000_3", "165.9998001"),
    ("SNV-ARC-4", "SNV_bor_464000000000_4", "165.9998001"),
    ("SNV-ARC-5", "SNV_bor_464000000000_5", "165.9998001"),
    ("SNV-DRC-1", "SNV_bor_402000000000", "165.9998001"),
    ("PNFP-ADV-1", "BOR_69_1", "165.9998001"),
    ("PNFP-ADV-2", "BOR_495_2", "165.9998001"),
    ("PNFP-ADV-3", "BOR_496_3", "165.9998001"),
    ("PNFP-ADV-4", "BOR_498_4", "165.9998001"),
    ("PNFP-ADV-5", "BOR_499_5", "165.9998001"),
    ("PNFP-ADV-6", "BOR_505_6", "165.9998001"),
    ("PNFP-ADV-7", "BOR_507_7", "165.9998001"),
    ("PNFP-ADV-8", "BOR_509_8", "165.9998001"),
    ("PNFP-ADV-9", "BOR_513_9", "165.9998001"),
    ("SNV-HC-1", "SNV_bor_55555SBD5", "D7B.0008087"),
    ("SNV-BANK-1", "SNV_bor_55555SBD8", "165.9998001"),
    ("SNV-TPS-1", "SNV_bor_55555TPS3", "FCT.8888"),
    ("SNV-HC-2", "SNV_bor_55555SBD9", "D7B.0008887"),
    ("SNV-BANK-2", "SNV_bor_55555SBD10", "165.9998001"),
    ("PNFP-TPS-1", "BOR_TPS 1_15", "P81.0004700"),
    ("PNFP-TPS-2", "BOR_TPS 2_16", "P81.0004700"),
    ("PNFP-TPS-3", "BOR_TPS 3_17", "P81.0004700"),
    ("PNFP-TPS-4", "BOR_TPS 4_18", "P81.0004700"),
    ("PNFP-TPS-5", "BOR_BNC Cap Trust I_19", "P81.0004700"),
    ("PNFP-TPS-6", "BOR_BNC Cap Trust II_20", "P81.0004700"),
    ("PNFP-TPS-7", "BOR_BNC Cap Trust III_21", "P81.0004700"),
    ("PNFP-TPS-8", "BOR_BNC Cap Trust IV_22", "P81.0004700"),
    ("PNFP-TPS-9", "BOR_Southcoast Capital Trust III_26", "D7B.0008087"),
    ("PNFP-TPS-10", "BOR_Valley Cap Trust I_23", "P81.0004700"),
    ("PNFP-TPS-11", "BOR_Valley Cap Trust II_24", "P81.0004700"),
    ("PNFP-TPS-12", "BOR_Valley Cap Trust III_25", "P81.0004700"),
    ("PNFP-HC-1", "BOR_SUB DEBT NOTE_14", "P81.0004700"),
    ("SNV-HC-3", "SNV_bor_55555SBD11", "D7B.0008887"),
    ("SNV-FFP-1", None, "165.9998001"),
]

# Define schema
schema = StructType([
    StructField("contract_id", StringType(), True),
    StructField("Sec_ID", StringType(), True),
    StructField("cost_center_id", StringType(), True)
])

# Create DataFrame
id_lookup_df = spark.createDataFrame(id_lookup_data, schema)

# Create temp view
id_lookup_df.createOrReplaceTempView("id_lookup")

In [0]:

# Source workbook and source tab (folder and destination are set by the widgets in Imports & Config)
source_file = "Wholesale Funding Dashboard.xlsx"
file_location = f"file:{source_folder}/{source_file}"
source_sheet = "Historical"   # tab that holds the source data


# Map source CSV headers to lowercase snake_case table columns
column_mapping = {
    "As Of Date": "as_of_date",
    "Entity": "entity",
    "Contract ID": "contract_id",
    "Tranche": "tranche",
    "Primary Instrument Type": "primary_instrument_type",
    "Sub-Instrument Type": "sub_instrument_type",
    "Product Type": "product_type",
    "Counterparty Name (if applicable)": "counterparty_name",
    "Account #": "account_number",
    "CUSIP": "cusip",
    "Issuance Date": "issuance_date",
    "Maturity Date": "maturity_date",
    "Months Remaining": "months_remaining",
    "Notice Date": "notice_date",
    "First Call Date": "first_call_date",
    "Contractual Maturity": "contractual_maturity",
    "Auto-roll @ Notice": "auto_roll_at_notice",
    "Rate Type": "rate_type",
    "Benchmark": "benchmark",
    "Base Rate": "base_rate",
    "Spread": "spread",
    "Coupon Rate": "coupon_rate",
    "Annualized Exp Coupon": "annualized_exp_coupon",
    "Broker Fee (CD only)": "broker_fee_cd_only",
    "All in Rate": "all_in_rate",
    "Annualized Exp All-In": "annualized_exp_all_in",
    "Accrual Convention": "accrual_convention",
    "Int. Pay Frequency": "interest_payment_frequency",
    "Monthly Expense (est.)": "monthly_expense_est",
    "Annualized Expense (est.)": "annualized_expense_est",
    "Est. Yield": "estimated_yield",
    "Target ($MM)": "target_millions",
    "Max ($MM)": "max_millions",
    "Principal Balance ($MM)": "current_balance",
    "Contract Balance ($MM)": "contract_balance",
    "Balance GL": "gl_account_number",
    "Income GL": "income_gl",
    "Balance Data Source:": "balance_data_source",
}


def clean_header(header):
    """
    Clean source headers by removing HTML breaks and standardizing spacing.
    """
    header = re.sub(r"<br\s*/?>", "", header, flags=re.IGNORECASE)
    header = " ".join(header.split())
    return header.strip()


# Read Excel without inferring data types.
# This protects IDs, account numbers and GL values from being converted incorrectly.
df = (
    spark.read.format('excel')
    .option('header', 'false')
    .option('inferSchema', "false")
    .option('dataAddress', source_sheet)
    .load(file_location)
)

# The excel format reader does not honor the header option on this compute;
# extract headers from the first row manually.
first_row = df.first()
headers = [str(first_row[i]) for i in range(len(df.columns))]
df = df.toDF(*headers)
df = df.filter(f.col(headers[0]) != first_row[0])


# Clean source headers.
cleaned_columns = [
    clean_header(column)
    for column in df.columns
]

if len(cleaned_columns) != len(set(cleaned_columns)):
    raise ValueError(
        "Duplicate column names were found after header cleanup."
    )

df = df.toDF(*cleaned_columns)


# Validate that source columns exist before renaming.
source_columns = set(df.columns)

missing_columns = [
    source_column
    for source_column in column_mapping
    if source_column not in source_columns
]


if missing_columns:
    raise ValueError(
        "Missing expected CSV columns after cleanup:\n"
        + "\n".join(f"  - {column}" for column in missing_columns)
        + "\n\nAvailable source columns:\n"
        + "\n".join(f"  - {column}" for column in sorted(source_columns))
    )


# Rename columns.
for source_column, target_column in column_mapping.items():
    if source_column in df.columns and source_column != target_column:
        df = df.withColumnRenamed(
            source_column,
            target_column
        )


# Cast Excel serial-number date columns to proper DateType.
# Excel stores dates as days since 1899-12-30.
date_columns = [
    "as_of_date",
    "issuance_date",
    "maturity_date",
    "notice_date",
    "first_call_date",
]

for col_name in date_columns:
    df = df.withColumn(
        col_name,
        f.coalesce(
            f.expr(f"try_to_date({col_name}, 'M/d/yy')"),
            f.expr(f"date_add('1899-12-30', try_cast(regexp_replace({col_name}, '[%,]', '') as int))")
        )
    )


# Cast numeric columns to DoubleType.
numeric_columns = [
    "months_remaining",
    "base_rate",
    "spread",
    "coupon_rate",
    "all_in_rate",
    "annualized_exp_coupon",
    "annualized_exp_all_in",
    "annualized_expense_est",
    "estimated_yield",
    "target_millions",
    "max_millions",
]

for col_name in numeric_columns:
    df = df.withColumn(
        col_name,
        f.expr(f"try_cast(regexp_replace(trim({col_name}), '[%,]', '') as double)")
    )

# Balance columns are whole-dollar amounts: round to the nearest dollar and store as BIGINT
# (avoids scientific notation like 2.258614302E9 that DOUBLE produces).
bigint_columns = [
    "current_balance",
    "contract_balance",
]

for col_name in bigint_columns:
    df = df.withColumn(
        col_name,
        f.expr(f"cast(round(try_cast(regexp_replace(trim({col_name}), '[$%,]', '') as double), 0) as bigint)")
    )
# Drop source columns that will be replaced with calculated versions.
columns_to_drop = [
    col for col in df.columns
    if col in (
        "Internal Flag", "Secured Flag", "Issuer Currency",
        "Optionalilty Type", "Settlement Type", "Entity ID",
        "DeptCode", "Sec_ID",
    )
]
df = df.drop(*columns_to_drop)


# --- Calculated columns ---
# deposit_id and cost_center_id: join to id_lookup temp view on contract_id
lookup_df = spark.table("id_lookup")
df = df.join(lookup_df, on="contract_id", how="left")

# Join dim_dates on as_of_date for calendar attributes.
dim_dates_df = spark.table("finance.default.dim_dates").select("Date", "Last_Business_Day_Of_Month", "Quarter_Year", "End_of_Month")
df = df.join(dim_dates_df, df["as_of_date"] == dim_dates_df["Date"], "left").drop("Date")

# Keep only month-end snapshots: as_of_date must equal the last business day of its month.
df = df.filter(f.col("as_of_date") == f.col("Last_Business_Day_Of_Month"))


# Remaining calculated columns
df = df.withColumns({
    "cusip": f.when(f.lower(f.col("cusip")).isin("n.a.", "na", "n/a", "none"), None).otherwise(f.col("cusip")),
    # Account numbers: trim, and null out placeholders such as '-', 'n.a.', blanks.
    "account_number": f.when(
        f.lower(f.trim(f.col("account_number"))).isin("-", "", "n.a.", "na", "n/a", "none"), None
    ).otherwise(f.trim(f.col("account_number"))),
    "counterparty_type": f.when(
        f.col("product_type").isin(
            "Adjustable Rate Credit", "Daily Rate Credit",
            "Fixed Rate Advance", "Floating Rate Advance", "Fed Funds Purchased"
        ), f.lit("Government Sponsored Entity")
    ).when(
        f.col("primary_instrument_type").isin(
            "Brokered Deposit"
        ), f.lit("Retail")
    ),
                        
    "currency_type": f.lit("USD"),
    "optionality_type": f.when(f.col("first_call_date").isNull(), f.lit(None)).otherwise(f.lit("Call")),
    "settlement_type": f.lit("Bilateral"),
    "secured_flag": f.when(f.col("primary_instrument_type") == "FHLB Advance", f.lit("Secured")).otherwise(f.lit("Unsecured")),
    "internal_flag": f.lit("FALSE"),
    "internal_transaction_flag": f.lit("FALSE"),
    "legal_entity_id": (
        f.when(f.col("cost_center_id").startswith("165"), f.lit("Bank"))
        .when(f.col("contract_id").startswith("PNFP-TPS-9"), f.lit("TPS"))
         .when(f.col("cost_center_id").startswith("D7B"), f.lit("HoldCo"))
         .when(f.col("cost_center_id").startswith("P81"), f.lit("TPS"))
         .when(f.col("cost_center_id").startswith("FCT"), f.lit("TPS"))
        ),
    "benchmark": f.when(
        f.lower(f.col("rate_type")) == "fixed", f.lit(None)
    ).otherwise(
        f.when(f.lower(f.col("benchmark")).isin("n.a.", "na", "n/a", "none"), None)
        .otherwise(f.col("benchmark"))
    ),
    "contractual_maturity": f.when(
        f.lower(f.col("contractual_maturity")).isin("n.a.", "na", "n/a", "none"), None
    ).otherwise(f.initcap(f.col("contractual_maturity"))),
    "auto_roll_at_notice": f.when(
        f.lower(f.col("auto_roll_at_notice")).isin("n.a.", "na", "n/a", "none"), None
    ).otherwise(f.initcap(f.col("auto_roll_at_notice"))),
    "gl_bs_acct": f.split(f.col("gl_account_number"), "/").getItem(0),
    "gl_bs_fee_acct": f.when(
        f.col("gl_account_number").contains("/"),
        f.split(f.col("gl_account_number"), "/").getItem(1)
    ),
})


# --- Internal (Second Leg) for Trust Preferred Securities ---
# Duplicate TPS rows with internal_flag / internal_transaction_flag = TRUE and legal_entity_id = HoldCo.
tps_internal = (
    df.filter(f.col("product_type") == "Trust Preferred Securities")
      .withColumn("internal_flag", f.lit("TRUE"))
      .withColumn("internal_transaction_flag", f.lit("TRUE"))
      .withColumn("legal_entity_id", f.lit("HoldCo"))
)
df = df.unionByName(tps_internal)


# Add ingestion metadata.
df = (
    df.withColumn("_ingested_at", f.current_timestamp())
      .withColumn("_source_file", f.lit(file_location))
)


# --- Reorder columns into logical groups ---
df = df.select(
    # ── Identifiers & Entity ──
    "contract_id",
    "Sec_ID",
    "entity",
    "legal_entity_id",
    "cost_center_id",
    "tranche",

    # ── Dates & Time ──
    "as_of_date",
    "End_of_Month",          # calendar month-end for the as_of_date
    "Quarter_Year",
    "issuance_date",
    "maturity_date",
    "months_remaining",
    "notice_date",
    "first_call_date",

    # ── Instrument Classification ──
    "primary_instrument_type",
    "sub_instrument_type",
    "product_type",

    # ── Counterparty & Account ──
    "counterparty_name",
    "counterparty_type",
    "account_number",
    "cusip",

    # ── Contract Terms ──
    "contractual_maturity",
    "auto_roll_at_notice",
    "optionality_type",
    "settlement_type",
    "secured_flag",
    "currency_type",

    # ── Rate Details ──
    "rate_type",
    "benchmark",
    "base_rate",
    "spread",
    "coupon_rate",
    "all_in_rate",
    "accrual_convention",
    "interest_payment_frequency",
    "broker_fee_cd_only",

    # ── Balances & Financials ──
    "current_balance",
    "contract_balance",
    "target_millions",
    "max_millions",
    "estimated_yield",
    "annualized_exp_coupon",
    "annualized_exp_all_in",
    "monthly_expense_est",
    "annualized_expense_est",

    # ── GL & Accounting ──
    
    "gl_bs_acct",
    "gl_bs_fee_acct",
    "income_gl",
    "balance_data_source",

    # ── Flags ──
    "internal_flag",
    "internal_transaction_flag",

    # ── Metadata ──
    "_ingested_at",
    "_source_file",
)


# Validate target column names are unique.
if len(df.columns) != len(set(df.columns)):
    raise ValueError(
        "Duplicate target column names were created during renaming."
    )


# Write output based on widget configuration.
if view_type == "table":
    writer = df.write.format("delta").mode(write_mode)
    if write_mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")
    writer.saveAsTable(target_name)
else:
    # Temp views are session-scoped and cannot be appended to; they are always replaced.
    if write_mode == "append":
        print("Note: append is not supported for temp views - the view will be replaced.")
    df.createOrReplaceTempView(target_name)


# Final validation.
result_df = spark.table(target_name)

row_count = result_df.count()
column_count = len(result_df.columns)

print(f"{view_type.title()} written successfully: {target_name} ({environment}, {write_mode})")
print(f"Rows in target: {row_count:,}")
print(f"Columns loaded: {column_count}")

print("\nFinal columns:")
for column_name in result_df.columns:
    print(f"  - {column_name}")

In [0]:
# # Validation: every as_of_date should be a month-end business day; check account_number placeholders are gone
# chk = spark.table(target_name)
# display(
#     chk.groupBy("as_of_date", "Last_Business_Day_Of_Month")
#        .agg(f.count("*").alias("rows"), f.sum("current_balance").alias("balance_mm"))
#        .orderBy("as_of_date")
# )
# print("Rows where as_of_date != last business day:",
#       chk.filter(f.col("as_of_date") != f.col("Last_Business_Day_Of_Month")).count())
# print("Rows with placeholder account numbers:",
#       chk.filter(f.trim(f.col("account_number")).isin("-", "")).count())
# print("Rows with null account_number:", chk.filter(f.col("account_number").isNull()).count())

In [0]:
from pyspark.sql import functions as f

# ======================================================================================
# WHOLESALE FUNDING
# ======================================================================================

wf = spark.table("finance.reporting.wholesalefunding")

latest_date = wf.select(
    f.max("as_of_date")
).collect()[0][0]

wf = wf.filter(
    f.col("as_of_date") == latest_date
)

# ======================================================================================
# GL REFERENCE (ACCOUNT NAME LOOKUP)
# ======================================================================================

gl_ref = (
    spark.table("silver_prod.reference_data.d_gl_account")
    .select(
        f.col("`Account ID`"),
        f.col("`Level 5 Desc`")
    )
)

gl_bs = (
    gl_ref.select(
        f.col("`Account ID`").alias("_bs_id"),
        f.col("`Level 5 Desc`").alias("gl_bs_acct_name")
    )
)

wf = (
    wf.join(
        gl_bs,
        wf["gl_bs_acct"] == gl_bs["_bs_id"],
        "left"
    )
    .drop("_bs_id")
)

# ======================================================================================
# WHOLESALE FUNDING BALANCES
# ======================================================================================

wf_recon = (
    wf.groupBy(
        "gl_bs_acct",
        "gl_bs_acct_name"
    )
    .agg(
        f.sum("current_balance").alias("wf_balance")
    )
)

# ======================================================================================
# TM1 TRIAL BALANCE
# ======================================================================================

tb = (
    spark.table("silver_prod.p021.tm1_financetrialbalancepbi")
    .filter(
        f.col("PeriodDate") == latest_date
    )
)

# If needed, uncomment and adjust:
#
# tb = tb.filter(
#     f.col("Versions") == "Actual"
# )
#
# tb = tb.filter(
#     f.col("FinanceMeasures") == "Balance"
# )

tb_recon = (
    tb.groupBy("Accounts")
    .agg(
        f.sum("VValue").alias("gl_balance"),
        f.concat_ws(
            ", ",
            f.sort_array(
                f.collect_set("Entities")
            )
        ).alias("entities")
    )
)

# ======================================================================================
# RECONCILIATION
# ======================================================================================

recon_df = (
    wf_recon.alias("wf")
    .join(
        tb_recon.alias("tb"),
        f.col("wf.gl_bs_acct").cast("string")
        == f.col("tb.Accounts").cast("string"),
        "full"
    )
    .select(
        f.coalesce(
            f.col("wf.gl_bs_acct").cast("string"),
            f.col("tb.Accounts").cast("string")
        ).alias("gl_account"),

        f.col("wf.gl_bs_acct_name"),

        f.coalesce(
            f.col("tb.gl_balance"),
            f.lit(0)
        ).alias("gl_balance"),

        f.coalesce(
            f.col("wf.wf_balance"),
            f.lit(0)
        ).alias("wf_balance"),

        (
            f.coalesce(f.col("wf.wf_balance"), f.lit(0))
            - f.coalesce(f.col("tb.gl_balance"), f.lit(0))
        ).alias("variance"),

        f.col("tb.entities")
    )
    .withColumn(
        "abs_variance",
        f.abs(f.col("variance"))
    )
    .orderBy(
        f.desc("abs_variance")
    )
)

# ======================================================================================
# DISPLAY
# ======================================================================================

display(recon_df)

In [0]:
from pyspark.sql import functions as f

# ======================================================================================
# LATEST WHOLESALE FUNDING SNAPSHOT
# ======================================================================================

wf = spark.table("finance.reporting.wholesalefunding")

latest_date = (
    wf.select(
        f.max("as_of_date")
    ).collect()[0][0]
)

wf = wf.filter(
    f.col("as_of_date") == latest_date
)

# ======================================================================================
# GL ACCOUNT REFERENCE
# ======================================================================================

gl_ref = (
    spark.table("silver_prod.reference_data.d_gl_account")
    .select(
        f.col("`Account ID`"),
        f.col("`Level 5 Desc`")
    )
)

gl_bs = (
    gl_ref.select(
        f.col("`Account ID`").alias("_bs_id"),
        f.col("`Level 5 Desc`").alias("gl_bs_acct_name")
    )
)

wf = (
    wf.join(
        gl_bs,
        wf["gl_bs_acct"] == gl_bs["_bs_id"],
        "left"
    )
    .drop("_bs_id")
)

# ======================================================================================
# WHOLESALE FUNDING BALANCES
# ======================================================================================

wf_recon = (
    wf.groupBy(
        "gl_bs_acct",
        "gl_bs_acct_name"
    )
    .agg(
        f.sum("current_balance").alias("wf_balance")
    )
)

# ======================================================================================
# TM1 TRIAL BALANCE
# ======================================================================================

tm1_date = latest_date.strftime("%Y%m%d")

tb = (
    spark.table("silver_prod.p021.tm1_financetrialbalancepbi")
    .filter(
        f.col("PeriodDate") == tm1_date
    )
)

tb_recon = (
    tb.groupBy("Accounts")
    .agg(
        f.sum("VValue").alias("gl_balance"),
        f.concat_ws(
            ", ",
            f.sort_array(
                f.collect_set("Entities")
            )
        ).alias("entities")
    )
)

# ======================================================================================
# RECONCILIATION
# ======================================================================================

recon_df = (
    wf_recon.alias("wf")
    .join(
        tb_recon.alias("tb"),
        f.col("wf.gl_bs_acct").cast("string")
        == f.col("tb.Accounts").cast("string"),
        "full"
    )
    .select(
        f.coalesce(
            f.col("wf.gl_bs_acct").cast("string"),
            f.col("tb.Accounts").cast("string")
        ).alias("gl_account"),

        f.col("wf.gl_bs_acct_name"),

        f.coalesce(
            f.col("tb.gl_balance"),
            f.lit(0)
        ).alias("gl_balance"),

        f.coalesce(
            f.col("wf.wf_balance"),
            f.lit(0)
        ).alias("wf_balance"),

        (
            f.coalesce(
                f.col("wf.wf_balance"),
                f.lit(0)
            )
            - f.coalesce(
                f.col("tb.gl_balance"),
                f.lit(0)
            )
        ).alias("variance"),

        f.col("tb.entities")
    )
    .withColumn(
        "abs_variance",
        f.abs(
            f.col("variance")
        )
    )
    .orderBy(
        f.desc("abs_variance")
    )
)

# ======================================================================================
# DISPLAY RESULTS
# ======================================================================================

display(recon_df)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW pnfp_wf_deposits AS
SELECT
  ps.load_date asofdate,
  d.end_of_month AS eomdate,
  ps.account_type,
  ps.account_number,
  CONCAT(ps.account_type, ps.account_number) as accountid,
  ps.customer_name,
  ps.product_type acct_typ_cde,
  ps.product_description acct_typ_desc,
  ps.balance,
  ps.date7_entered as open_date,
  ps.maturity_date7 AS maturity_date
FROM
  silver_prod.strategicfinance.pnfp_stacked_data ps
    JOIN dim_dates d
      ON d.date = PS.LOAD_DATE
WHERE
  ps.product_type IN (
    '3Z', '4E', '3I', '7G', 'IM', '30', '6I', '7I', '9I', '28', '5I', '4I'
  )
  AND ps.account_type <> 'L'
  AND (
    RIGHT(Product_Description, 1) = 'B'
    OR (
      ps.product_type IN ('4E', '3I', '3Z')
      AND account_type <> 'T'
    )
  )
  AND d.date = d.last_business_day_of_month
  AND LOAD_DATE = '2026-09-30'

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW SNV_brokered_cds AS
SELECT
  d.last_business_day_of_month asofdate,
  d.end_of_month as eomdate,
  'C' as account_type,
  cusip as account_number,
  CONCAT('165', cusip) as accountid,
  CounterParty as customer_name,
  '092' as acct_typ_cde,
  'Brokered CD' as acct_typ_desc,
  FairValue as balance,
  SettlementDate as open_date,
  StatedMaturityDate as maturity_date

FROM
  silver_dev.tpg_reports_pnfp.position_report
      JOIN dim_dates d
      ON d.date = asofdate
WHERE
  asset_liability_flag = 'L'
  AND d.date = d.last_business_day_of_month
    AND D.DATE = '2026-09-30'

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW snv_wf_deposits AS
SELECT
  dm.prcs_dte as asofdate,
  d.end_of_month as eomdate,
  df.dp_cat_cde as account_type,
  df.acct_grp_nbr as account_number,
  CONCAT(df.PRCS_GRP_NBR, df.acct_grp_nbr) as accountid,
  df.ACCT_CUST_SHRT_NME customer_name,
  df.acct_typ_cde,
  df.acct_typ_desc,
  dm.lgr_bal_amt as balance,
  df.eff_dte as open_date,
  CAST(NULL AS DATE) AS maturity_date
FROM
  silver_prod.dwdx1.tb_dp_dyy_c2_acct_scd_dim df
    JOIN silver_prod.dwdx1.tb_dp_sz9_dp_acct_d_fact dm
      ON dm.DW_PROD_SERV_ID = df.DW_PROD_SERV_ID
    JOIN dim_dates d
      ON d.date = dm.PRCS_DTE
WHERE
  df.acct_typ_cde IN ('439', '440', '196', '194', '199')
  AND d.date = d.last_business_day_of_month
  AND D.DATE = '2026-09-30'

In [0]:
# Stack PNFP deposits, PNFP brokered CDs and SNV wholesale funding deposits into one view
spark.sql("""
CREATE OR REPLACE TEMP VIEW brokered_src AS
SELECT 'PNFP' AS legacy_bank, 'JH' AS source, * FROM pnfp_wf_deposits
UNION ALL
SELECT 'SNV' AS legacy_bank, 'SNV TPG' AS source, * FROM snv_brokered_cds
UNION ALL
SELECT 'SNV' AS legacy_bank, 'FIS' AS source, * FROM snv_wf_deposits
""")


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW wf_brokered_deposits AS
SELECT DISTINCT
*
FROM
  finance.reporting.wholesalefunding
WHERE
  primary_instrument_type = 'Brokered Deposit'
  AND as_of_date = '2026-09-30'
GROUP BY ALL;

CREATE OR REPLACE TEMP VIEW wf_brokered_deposits_nulls AS
SELECT DISTINCT
contract_id,
contract_balance
FROM
  finance.reporting.wholesalefunding
WHERE
  primary_instrument_type = 'Brokered Deposit'
  AND as_of_date = '2026-09-30'
GROUP BY ALL;

CREATE OR REPLACE TEMP VIEW wf_deposits_enriched AS
SELECT
DISTINCT
  d.*,
  b.max_millions,
  b.target_millions,
  B.CONTRACT_BALANCE,
  b.cusip AS brokered_cusip,
  b.contract_id AS brokered_contract_id,
  b.tranche AS brokered_tranche,
  b.account_number as ann


FROM
wf_brokered_deposits b
  LEFT JOIN  brokered_src d
    ON d.account_number = b.account_number
    OR d.account_number = b.account_number 
    OR RIGHT(d.account_number,10) = b.account_number

    
    where b.contract_id IN (

SELECT DISTINCT
contract_id
FROM
  finance.reporting.wholesalefunding
WHERE
  primary_instrument_type = 'Brokered Deposit'
  AND account_number IS NULL
  AND as_of_date = '2026-09-30'
)

    
    
    ;

SELECT * FROM wf_deposits_enriched


-- CASE WHEN 'NMD BROK' 

-- BrokNMD-(CONTRACT INUMBER)-tranche

-- CASE WHEN 'NMD BROK TERM'  (THOSE W MAT DAYS)

-- BrokTerm-(CONTRACT INUMBER)-tranche

-- CASE WHEN 'Brok CD' 

-- BrokCD-(CONTRACT INUMBER)-tranche


-- if tranche is null then make it 1 ..

-- src_balance,
-- allocated_balance,



select 

brokered_contract_id,
brokered_tranche,


FROM wf_deposits_enriched



In [0]:
%sql
SELECT * from finance.reporting.wholesalefunding WHere as_of_date = '2026-09-30' and contract_id = 'SNV-NMD-1' group by all


--take out $8m from 
--Add a row  ($8.5M goes into row) 800106466618
-- 800106466634 go to the others 

-- Ensure legal entity 


In [0]:
%sql

SELECT *, 

    CASE WHEN product_type = 'Brokered Deposit' 
        THEN 'Y'
            END as deposit_flag
            
    CASE WHEN product_type = 'Brokered Deposit' 
        THEN 'Y'  W  
        
        
        
FROM finance.reporting.wholesalefunding WHERE as_of_date = '2026-09-30' group by all



--Deposit_Flag


--Deposit_Source (FIS, JH, Hybrid, Mannual)
o     --Go into PNFP Stacked,
        -- Go into SNV FIS tables
        --Brokered SNV in TPG (get into from these)

        


--Deposit_ID (Unique Account Number from source system)

--Create a notebook that 1)YTD Wholesale Funding Table 2)WHSF TO GL Recon 3) Trended View / Metrics 4) ReadMe/Config/Mapping/Assumptions





--
--

In [0]:
%sql

SELECT DISTINCT cusip, primary_instrument_type,
 gl_bs_acct, entity, legal_entity_id, sum(contract_balance) FROM finance.reporting.wholesalefunding WHERE as_of_date = '2026-09-30' group by all

In [0]:
 %sql
SELECT
  dim_dates.last_business_day_of_month as asofdate,
  regexp_replace(tm1.projects, 'No Project', '') as legacy_bank,
  dim_dates.date as eomdate,
  gl.`Level 3 ID` as level3_id,
  gl.`Level 6 ID` as level6_id,
  gl.`Level 7 ID` as level7_id,
  gl.`Account Desc` as gl_bs_acct_name,
  gl.`Account ID` as gl_bs_acct,
  SUM(CAST(tm1.vvalue AS DOUBLE)) as gl_balance,
  tm1.Entities as entity,
  -- e.entity_desc,
  e.Parent1,
  e.LOB_LVL05_node,
    e.LOB_LVL05_description

  -- gl.`Z_Account Attributes.ReconType` as Recon_Type,
  -- gl.`Z_Account Attributes.ReconGroup` as Recon_Group
FROM
  silver_prod.p021.tm1_financetrialbalancepbi tm1
    JOIN silver_prod.reference_data.d_gl_account gl
      ON tm1.Accounts = gl.`Account ID`
    JOIN dim_dates
      ON dim_dates.date = TO_DATE(tm1.prcs_dte, 'yyyyMMdd')
    LEFT JOIN silver_prod.reference_data.d_entity_hierarchy_wide e
      ON e.entity_id = tm1.Entities
WHERE
  Versions = 'ACTUAL'
  AND DataSources = 'TotalCur'
  AND FinanceMeasures = 'EndBal'
  AND RIGHT(Periods, 3) = 'YTD'
  AND dim_dates.date = '2026-09-30'

  AND (
    gl.`Level 6 ID` = 'TOTBROKE'
    OR gl.`Account ID` = '2411007'
        OR gl.`Account Desc` = 'Due To FHLB - Long Term'
        OR gl.`Account ID` = '2411009'
    
    OR gl.`Level 5 ID` = 'NOTESPAY'
    
  )
  AND dim_dates.date = dim_dates.end_of_month
GROUP BY
  ALL


In [0]:
a = 1397112042.44 -  1397739872.55 
print(format(a, ',.2f'))

In [0]:
 %sql
SELECT
  dim_dates.last_business_day_of_month as asofdate,
  regexp_replace(tm1.projects, 'No Project', '') as legacy_bank,
  dim_dates.date as eomdate,
    gl.`Level 4 ID` as level4_id,
  gl.`Level 5 ID` as level5_id,
  gl.`Level 6 ID` as level6_id,
  gl.`Level 7 ID` as level7_id,
  gl.`Account Desc` as gl_bs_acct_name,
  gl.`Account ID` as gl_bs_acct,
  SUM(CAST(tm1.vvalue AS DOUBLE)) as gl_balance,
  tm1.Entities as entity,
  -- e.entity_desc,
  e.Parent1,
  e.LOB_LVL05_node,
    e.LOB_LVL05_description

  -- gl.`Z_Account Attributes.ReconType` as Recon_Type,
  -- gl.`Z_Account Attributes.ReconGroup` as Recon_Group
FROM
  silver_prod.p021.tm1_financetrialbalancepbi tm1
    JOIN silver_prod.reference_data.d_gl_account gl
      ON tm1.Accounts = gl.`Account ID`
    JOIN dim_dates
      ON dim_dates.date = TO_DATE(tm1.prcs_dte, 'yyyyMMdd')
    LEFT JOIN silver_prod.reference_data.d_entity_hierarchy_wide e
      ON e.entity_id = tm1.Entities
WHERE
  Versions = 'ACTUAL'
  AND DataSources = 'TotalCur'
  AND FinanceMeasures = 'EndBal'
  and     gl.`Level 4 ID`  <>'TOTLOANS'
    and     gl.`Level 4 ID`  <>'FURNEQUIP'
        and     gl.`Level 4 ID`  <>'OTHASSET'
  AND RIGHT(Periods, 3) = 'YTD'
  AND dim_dates.date = '2026-09-30'
  and tm1.VValue > '284000000'
  AND dim_dates.date = dim_dates.end_of_month
GROUP BY
  ALL
2321008

In [0]:
%sql
SELECT
  *
FROM
  silver_prod.reference_data.d_entity_hierarchy_wide gl
WHERE
Entity_ID = 'P81.0004700'



In [0]:
import pandas as pd

# Latest as_of_date snapshot
wf = spark.table("finance.reporting.wholesalefunding")
latest_date = wf.select(f.max("as_of_date")).collect()[0][0]
wf = wf.filter(f.col("as_of_date") == latest_date)

# GL reference — Level 5 Desc for account names
gl_ref = spark.table("silver_prod.reference_data.d_gl_account").select(
    f.col("`Account ID`"),
    f.col("`Level 5 Desc`"),
)

# Join for balance GL name
gl_bs = gl_ref.select(
    f.col("`Account ID`").alias("_bs_id"),
    f.col("`Level 5 Desc`").alias("gl_bs_acct_name"),
)
wf = wf.join(gl_bs, wf["gl_bs_acct"] == gl_bs["_bs_id"], "left").drop("_bs_id")

# Join for fee GL name (when populated)
gl_fee = gl_ref.select(
    f.col("`Account ID`").alias("_fee_id"),
    f.col("`Level 5 Desc`").alias("gl_bs_fee_acct_name"),
)
wf = wf.join(gl_fee, wf["gl_bs_fee_acct"] == gl_fee["_fee_id"], "left").drop("_fee_id")

# Aggregate by primary instrument type, then by GL detail
summary_df = (
    wf.groupBy(
        "primary_instrument_type",
        "product_type",
        "gl_bs_acct", "gl_bs_acct_name",
        "gl_bs_fee_acct", "gl_bs_fee_acct_name",
    )
    .agg(
        f.sum("current_balance").alias("total_balance_mm"),
        (
            f.sum(f.col("estimated_yield") * f.col("current_balance"))
            / f.sum("current_balance")
        ).alias("wtd_avg_yield"),
    )
    .orderBy("primary_instrument_type", "product_type", "gl_bs_acct")
)

# Subtotals by primary instrument type
subtotals_df = (
    wf.groupBy("primary_instrument_type")
    .agg(
        f.sum("current_balance").alias("total_balance_mm"),
        (
            f.sum(f.col("estimated_yield") * f.col("current_balance"))
            / f.sum("current_balance")
        ).alias("wtd_avg_yield"),
    )
)

pdf = summary_df.toPandas()
sub_pdf = subtotals_df.toPandas().set_index("primary_instrument_type")

# --- Build styled HTML report with subtotal rows ---
rows = []
prev_instrument = None
for _, r in pdf.iterrows():
    instrument = r["primary_instrument_type"] or "\u2014"
    # Insert subtotal row when the instrument group changes
    if prev_instrument is not None and instrument != prev_instrument:
        st = sub_pdf.loc[prev_instrument]
        st_bal = f'${st["total_balance_mm"]:,.2f}' if pd.notna(st["total_balance_mm"]) else "\u2014"
        st_yld = f'{st["wtd_avg_yield"]:.4%}' if pd.notna(st["wtd_avg_yield"]) else "\u2014"
        rows.append(
            f'<tr class="sub"><td colspan="6">{prev_instrument} Subtotal</td>'
            f'<td class="r">{st_bal}</td><td class="r">{st_yld}</td></tr>'
        )
    prev_instrument = instrument

    product   = r["product_type"] or "\u2014"
    bs_acct   = r["gl_bs_acct"] or "\u2014"
    bs_name   = r["gl_bs_acct_name"] or "\u2014"
    fee_acct  = r["gl_bs_fee_acct"] if pd.notna(r["gl_bs_fee_acct"]) else "\u2014"
    fee_name  = r["gl_bs_fee_acct_name"] if pd.notna(r["gl_bs_fee_acct_name"]) else "\u2014"
    bal       = f'${r["total_balance_mm"]:,.2f}' if pd.notna(r["total_balance_mm"]) else "\u2014"
    yld       = f'{r["wtd_avg_yield"]:.4%}' if pd.notna(r["wtd_avg_yield"]) else "\u2014"
    rows.append(
        f"<tr><td>{instrument}</td><td>{product}</td>"
        f"<td>{bs_acct}</td><td>{bs_name}</td>"
        f"<td>{fee_acct}</td><td>{fee_name}</td>"
        f'<td class="r">{bal}</td><td class="r">{yld}</td></tr>'
    )

# Final subtotal for the last group
if prev_instrument is not None and prev_instrument in sub_pdf.index:
    st = sub_pdf.loc[prev_instrument]
    st_bal = f'${st["total_balance_mm"]:,.2f}' if pd.notna(st["total_balance_mm"]) else "\u2014"
    st_yld = f'{st["wtd_avg_yield"]:.4%}' if pd.notna(st["wtd_avg_yield"]) else "\u2014"
    rows.append(
        f'<tr class="sub"><td colspan="6">{prev_instrument} Subtotal</td>'
        f'<td class="r">{st_bal}</td><td class="r">{st_yld}</td></tr>'
    )

# Grand total
grand_bal = pdf["total_balance_mm"].sum()
grand_yld = (
    (pdf["wtd_avg_yield"] * pdf["total_balance_mm"]).sum() / grand_bal
    if grand_bal else 0
)
rows.append(
    f'<tr class="grand"><td colspan="6">Grand Total</td>'
    f'<td class="r">${grand_bal:,.2f}</td><td class="r">{grand_yld:.4%}</td></tr>'
)

html = f"""
<style>
  .wf {{ border-collapse:collapse; width:100%; font-family:'Segoe UI',Arial,sans-serif; font-size:13px; }}
  .wf th {{ background:#1a3c5e; color:#fff; padding:10px 8px; text-align:left; position:sticky; top:0; }}
  .wf td {{ padding:8px; border-bottom:1px solid #e2e8f0; }}
  .wf tr:nth-child(even) {{ background:#f8fafc; }}
  .wf tr:hover {{ background:#e2e8f0; }}
  .wf .r {{ text-align:right; }}
  .wf .sub td {{ background:#e2e8f0; font-weight:600; border-top:2px solid #94a3b8; }}
  .wf .grand td {{ background:#1a3c5e; color:#fff; font-weight:700; border-top:3px solid #0f172a; }}
  .wf-h {{ font-family:'Segoe UI',Arial,sans-serif; color:#1a3c5e; margin-bottom:4px; }}
  .wf-s {{ font-family:'Segoe UI',Arial,sans-serif; color:#64748b; font-size:14px; margin-bottom:16px; }}
</style>
<h2 class="wf-h">Wholesale Funding Summary</h2>
<div class="wf-s">Balance &amp; Weighted Avg Yield by Primary Instrument &amp; GL Account &mdash; As of {latest_date}</div>
<table class="wf">
  <thead><tr>
    <th>Primary Instrument</th><th>Product Type</th>
    <th>GL BS Acct</th><th>GL BS Acct Name</th>
    <th>GL Fee Acct</th><th>GL Fee Acct Name</th>
    <th class="r">Total Balance ($MM)</th><th class="r">Wtd Avg Yield</th>
  </tr></thead>
  <tbody>{''.join(rows)}</tbody>
</table>
"""

displayHTML(html)

# Save a copy of the report to Outputs/html when the Save HTML Report widget is 'yes'
if save_html:
    import os
    os.makedirs(html_output_folder, exist_ok=True)
    html_file = os.path.join(
        html_output_folder,
        f"wholesale_funding_summary_{str(latest_date).replace('-', '')}.html",
    )
    with open(html_file, "w", encoding="utf-8") as fh:
        fh.write(
            "<!DOCTYPE html>\n<html><head><meta charset='utf-8'>"
            f"<title>Wholesale Funding Summary - {latest_date}</title></head>"
            f"<body>{html}</body></html>"
        )
    print(f"HTML report saved: {html_file}")
else:
    print("HTML report not saved (Save HTML Report = no).")

# Save the same report as CSV (detail rows, instrument subtotals, grand total) to Outputs/csv
if save_csv:
    import os
    csv_cols = [
        "row_type", "primary_instrument_type", "product_type",
        "gl_bs_acct", "gl_bs_acct_name", "gl_bs_fee_acct", "gl_bs_fee_acct_name",
        "total_balance_mm", "wtd_avg_yield",
    ]
    csv_rows = []
    for instrument, grp in pdf.groupby("primary_instrument_type", dropna=False, sort=False):
        for _, r in grp.iterrows():
            csv_rows.append({"row_type": "Detail", **{c: r[c] for c in csv_cols[1:]}})
        if instrument in sub_pdf.index:
            st = sub_pdf.loc[instrument]
            csv_rows.append({
                "row_type": "Subtotal",
                "primary_instrument_type": f"{instrument} Subtotal",
                "total_balance_mm": st["total_balance_mm"],
                "wtd_avg_yield": st["wtd_avg_yield"],
            })
    csv_rows.append({
        "row_type": "Grand Total",
        "primary_instrument_type": "Grand Total",
        "total_balance_mm": grand_bal,
        "wtd_avg_yield": grand_yld,
    })
    csv_pdf = pd.DataFrame(csv_rows, columns=csv_cols)
    csv_pdf.insert(0, "as_of_date", latest_date)

    os.makedirs(csv_output_folder, exist_ok=True)
    csv_file = os.path.join(
        csv_output_folder,
        f"wholesale_funding_summary_{str(latest_date).replace('-', '')}.csv",
    )
    csv_pdf.to_csv(csv_file, index=False)
    print(f"CSV report saved: {csv_file} ({len(csv_pdf)} rows)")
else:
    print("CSV report not saved (Save CSV Report = no).")

In [0]:
import io, base64, os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Balance & rate trends by Product (primary_instrument_type) and Sub-Product (product_type),
# sourced from the 'Historical (2)' tab of the same workbook. Reuses column_mapping / clean_header
# from the Dataframe Creation cell.
hist_sheet = "Historical (2)"
month_end_only = True   # True = month-end snapshots (same rule as the main view); False = every as_of_date

h = (
    spark.read.format("excel")
    .option("header", "false").option("inferSchema", "false")
    .option("dataAddress", hist_sheet)
    .load(file_location)
)
h_first = h.first()
h = h.toDF(*[clean_header(str(x)) for x in h_first])
h = h.filter(f.col(h.columns[0]) != clean_header(str(h_first[0])))
for src, tgt in column_mapping.items():
    if src in h.columns:
        h = h.withColumnRenamed(src, tgt)

# Dates: 'M/d/yy' text or Excel serial numbers
h = h.withColumn(
    "as_of_date",
    f.coalesce(
        f.expr("try_to_date(as_of_date, 'M/d/yy')"),
        f.expr("date_add('1899-12-30', try_cast(regexp_replace(as_of_date, '[%,]', '') as int))"),
    ),
)

# Balance: strip commas. Rates: values like '3.87%' are divided by 100 so they match decimal values like 0.0387.
h = h.withColumn("current_balance", f.expr("try_cast(regexp_replace(trim(current_balance), ',', '') as double)"))
for c in ("estimated_yield", "all_in_rate"):
    h = h.withColumn(
        c,
        f.expr(
            f"case when {c} like '%\\%%' then try_cast(regexp_replace(trim({c}), '[%,]', '') as double) / 100 "
            f"else try_cast(regexp_replace(trim({c}), ',', '') as double) end"
        ),
    )

if month_end_only:
    dd = spark.table("finance.default.dim_dates").select("Date", "Last_Business_Day_Of_Month")
    h = (h.join(dd, h["as_of_date"] == dd["Date"], "left")
          .filter(f.col("as_of_date") == f.col("Last_Business_Day_Of_Month"))
          .drop("Date", "Last_Business_Day_Of_Month"))

h = h.withColumns({
    "product": f.coalesce(f.col("primary_instrument_type"), f.lit("(Unspecified)")),
    "sub_product": f.coalesce(f.col("product_type"), f.col("sub_instrument_type"), f.lit("(Unspecified)")),
})


def _trend(df, keys):
    """Balance ($MM) and balance-weighted avg yield / all-in rate per as_of_date and keys."""
    def wavg(c):
        w = f.when(f.col(c).isNotNull(), f.col("current_balance"))
        return f.try_divide(f.sum(f.col(c) * f.col("current_balance")), f.sum(w))  # NULL when balance is 0
    return (
        df.groupBy("as_of_date", *keys)
          .agg(
              (f.sum("current_balance") / 1e6).alias("balance_mm"),
              wavg("estimated_yield").alias("wtd_avg_yield"),
              wavg("all_in_rate").alias("wtd_avg_all_in_rate"),
          )
          .orderBy("as_of_date", *keys)
          .toPandas()
    )

prod_trend = _trend(h, ["product"])
sub_trend = _trend(h, ["product", "sub_product"])


def _fig_to_img(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    return f'<img src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}" style="max-width:100%"/>'


def _trend_fig(pdf_, series_col, title):
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
    for name, g in pdf_.groupby(series_col):
        axes[0].plot(g["as_of_date"], g["balance_mm"], marker="o", ms=3, label=name)
        axes[1].plot(g["as_of_date"], g["wtd_avg_yield"], marker="o", ms=3, label=name)
    axes[0].set_title(f"{title} — Balance ($MM)")
    axes[0].yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
    axes[1].set_title(f"{title} — Wtd Avg Est. Yield")
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=2))
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.tick_params(axis="x", rotation=30)
    axes[1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
    fig.tight_layout()
    return _fig_to_img(fig)


sections = ["<h3 class='wf-h'>All Products</h3>" + _trend_fig(prod_trend, "product", "By Product")]
for prod in sorted(sub_trend["product"].unique()):
    sections.append(
        f"<h3 class='wf-h'>{prod}</h3>"
        + _trend_fig(sub_trend[sub_trend["product"] == prod], "sub_product", f"{prod} by Sub-Product")
    )

# Latest vs first-period change table by sub-product
first_dt, last_dt = sub_trend["as_of_date"].min(), sub_trend["as_of_date"].max()
chg = (
    sub_trend[sub_trend["as_of_date"].isin([first_dt, last_dt])]
    .pivot_table(index=["product", "sub_product"], columns="as_of_date",
                 values=["balance_mm", "wtd_avg_yield"], aggfunc="sum")
)
chg_rows = []
for (prod, sub), r in chg.iterrows():
    b0, b1 = r.get(("balance_mm", first_dt)), r.get(("balance_mm", last_dt))
    y0, y1 = r.get(("wtd_avg_yield", first_dt)), r.get(("wtd_avg_yield", last_dt))
    fb = lambda v: f"${v:,.1f}" if pd.notna(v) else "\u2014"
    fy = lambda v: f"{v:.3%}" if pd.notna(v) else "\u2014"
    db = fb(b1 - b0) if pd.notna(b0) and pd.notna(b1) else "\u2014"
    dy = f"{(y1 - y0) * 1e4:+.1f} bps" if pd.notna(y0) and pd.notna(y1) else "\u2014"
    chg_rows.append(
        f"<tr><td>{prod}</td><td>{sub}</td><td class='r'>{fb(b0)}</td><td class='r'>{fb(b1)}</td>"
        f"<td class='r'>{db}</td><td class='r'>{fy(y0)}</td><td class='r'>{fy(y1)}</td><td class='r'>{dy}</td></tr>"
    )

freq_label = "month-end snapshots" if month_end_only else "all as-of dates"
hist_html = f"""
<style>
  .wf {{ border-collapse:collapse; width:100%; font-family:'Segoe UI',Arial,sans-serif; font-size:13px; }}
  .wf th {{ background:#1a3c5e; color:#fff; padding:8px; text-align:left; }}
  .wf td {{ padding:6px 8px; border-bottom:1px solid #e2e8f0; }}
  .wf tr:nth-child(even) {{ background:#f8fafc; }}
  .wf .r {{ text-align:right; }}
  .wf-h {{ font-family:'Segoe UI',Arial,sans-serif; color:#1a3c5e; margin:18px 0 4px; }}
  .wf-s {{ font-family:'Segoe UI',Arial,sans-serif; color:#64748b; font-size:14px; margin-bottom:12px; }}
</style>
<h2 class="wf-h">Wholesale Funding Historical Trends</h2>
<div class="wf-s">Balance &amp; balance-weighted yield by Product and Sub-Product &mdash; {first_dt} to {last_dt} ({freq_label}, source tab '{hist_sheet}')</div>
<h3 class="wf-h">Change by Sub-Product: {first_dt} &rarr; {last_dt}</h3>
<table class="wf"><thead><tr>
  <th>Product</th><th>Sub-Product</th><th class='r'>Balance {first_dt} ($MM)</th><th class='r'>Balance {last_dt} ($MM)</th>
  <th class='r'>&Delta; Balance ($MM)</th><th class='r'>Yield {first_dt}</th><th class='r'>Yield {last_dt}</th><th class='r'>&Delta; Yield</th>
</tr></thead><tbody>{''.join(chg_rows)}</tbody></table>
{''.join(sections)}
"""

displayHTML(hist_html)

# Save HTML / CSV copies (controlled by the Save HTML / Save CSV widgets)
stamp = str(last_dt).replace("-", "")
if save_html:
    os.makedirs(html_output_folder, exist_ok=True)
    p_html = os.path.join(html_output_folder, f"wholesale_funding_trends_{stamp}.html")
    with open(p_html, "w", encoding="utf-8") as fh:
        fh.write(f"<!DOCTYPE html><html><head><meta charset='utf-8'><title>Wholesale Funding Trends</title></head><body>{hist_html}</body></html>")
    print(f"Trend HTML saved: {p_html}")
if save_csv:
    os.makedirs(csv_output_folder, exist_ok=True)
    p_csv = os.path.join(csv_output_folder, f"wholesale_funding_trends_{stamp}.csv")
    sub_trend.to_csv(p_csv, index=False)
    print(f"Trend CSV saved: {p_csv} ({len(sub_trend)} rows)")

In [0]:
%sql

select * from finance.reporting.wholesalefunding